# 8) Demand response: price-responsive demand, scarcity and load shifting

## Objectives

At the end of this notebook you should be able to:

- model **price-responsive demand** — consumers who stop consuming above their willingness to pay — and show it is the same as a "generator" offering load reduction at that price
- explain **scarcity pricing**: what sets the price when supply is short, and why a model without demand response simply fails
- model **load shifting** (heat pumps, electric vehicles, flexible industry) as a lossless store of delayed demand, in your own model and in PyPSA
- compare the value of load shifting with the storage of lesson 7

Until now demand was a fixed number every hour and only supply could adapt. In reality some consumers respond to prices, and a growing share of demand — electric heating, vehicle charging — can move in time. This lesson adds both, first in an ordinary week (lesson 7's), then in a **stressed** one: a cold, still, cloudy week in which imports are cut, the coal plant is on outage and the reservoir is nearly dry. All numbers are **illustrative**.

In [1]:
import logging
import warnings

import linopy
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

### Lesson 7's week, and a stressed version of it

In [2]:
T = 168
h = np.arange(T)
hour, day = h % 24, h // 24
rng = np.random.default_rng(3)
load = (2000 + 300 * np.sin(np.pi * (hour - 6) / 12).clip(0) + 750 * np.exp(-((hour - 19) / 2.0) ** 2)
        - 150 * (day >= 5) + rng.normal(0, 30, T))
cloud = np.repeat([0.9, 0.6, 1.0, 0.8, 0.4, 1.0, 0.9], 24)
solar = 2500 * 0.75 * np.clip(np.sin(np.pi * (hour - 6) / 13), 0, None) * cloud
wind = 1000 * np.clip(0.3 + 0.25 * np.sin(2 * np.pi * h / 96), 0.02, 0.9)


def system(stress=False):
    """Thermal fleet, free energy (MW) and reservoir inflow (MW) of the normal or the stressed week."""
    thermal = pd.DataFrame({"p_nom": [350, 900, 600, 800, 400], "mc": [12.0, 89.1, 94.2, 104.1, 128.9]},
                           index=pd.Index(["nuclear", "ccgt", "coal", "imports", "ocgt"], name="gen"))
    free, inflow = solar + wind + 300.0, 120.0
    if stress:                                  # cloudy, still, imports cut, coal on outage, nearly dry
        free, inflow = 0.5 * solar + 0.2 * wind + 300.0, 40.0
        thermal.loc[["imports", "coal"], "p_nom"] = 0
    return thermal, free, inflow


firm = {s: system(s)[0].p_nom.sum() + 1000 for s in (False, True)}          # + the 1,000 MW reservoir turbine
print(f"peak demand {load.max():,.0f} MW | firm capacity: normal week {firm[False]:,.0f} MW, stressed week {firm[True]:,.0f} MW")
print(f"stressed week: free energy {system(True)[1].min():,.0f}-{system(True)[1].max():,.0f} MW, "
      f"reservoir inflow {system(True)[2] * T:,.0f} MWh")

peak demand 2,785 MW | firm capacity: normal week 4,050 MW, stressed week 2,650 MW
stressed week: free energy 310-1,326 MW, reservoir inflow 6,720 MWh


### Price-responsive demand

Two groups of consumers are willing to cut consumption when power gets expensive: an **industrial** load of 300 MW that stops above 300 €/MWh, and **commercial** loads of 250 MW that stop above 1,000 €/MWh. Everyone else is cut only as a last resort, valued at the **value of lost load (VOLL)**, here 3,000 €/MWh.

How do we put a consumer's choice into a cost-minimising model? A consumer who would pay at most 300 €/MWh *loses* 300 € of value for every MWh not consumed. So a reduction r of their demand costs the system 300 · r — exactly like a generator producing r at a marginal cost of 300 €/MWh. Minimising cost **including the value of lost consumption** is the same as maximising welfare (M2.1): demand response enters the model as extra "supply" offered at the willingness to pay.

In [3]:
DR = pd.DataFrame({"p_nom": [300, 250], "wtp": [300.0, 1000.0]}, index=pd.Index(["industry", "commercial"], name="dr"))
VOLL = 3000.0


def week(stress=False, dr=True, shift=None):
    """Own Linopy model. dr: price-responsive demand + VOLL shedding. shift: dict(p, e) for load shifting."""
    thermal, free, inflow = system(stress)
    m = linopy.Model()
    t, G = pd.RangeIndex(T, name="t"), thermal.index
    g = m.add_variables(lower=0, upper=xr.DataArray(np.tile(thermal.p_nom.to_numpy(), (T, 1)), coords=[t, G]), name="g")
    f = m.add_variables(lower=0, upper=xr.DataArray(free, coords=[t]), name="free")
    d = m.add_variables(lower=0, upper=1000, coords=[t], name="turbine")
    s = m.add_variables(lower=0, coords=[t], name="spill")
    e = m.add_variables(lower=0, upper=30_000, coords=[t], name="level")
    m.add_constraints(e - e.roll(t=1) + d + s == inflow, name="water")
    supply = g.sum("gen") + f + d
    cost = (g * xr.DataArray(thermal.mc.to_numpy(), coords=[G])).sum()
    if dr:
        D = DR.index
        r = m.add_variables(lower=0, upper=xr.DataArray(np.tile(DR.p_nom.to_numpy(), (T, 1)), coords=[t, D]), name="reduce")
        shed = m.add_variables(lower=0, coords=[t], name="shed")
        supply = supply + r.sum("dr") + shed
        cost = cost + (r * xr.DataArray(DR.wtp.to_numpy(), coords=[D])).sum() + VOLL * shed.sum()
    if shift:
        later = m.add_variables(lower=0, upper=shift["p"], coords=[t], name="delay")          # consume less now ...
        back = m.add_variables(lower=0, upper=shift["p"], coords=[t], name="catch_up")        # ... and more later
        backlog = m.add_variables(lower=0, upper=shift["e"], coords=[t], name="backlog")
        m.add_constraints(backlog - backlog.roll(t=1) - later + back == 0, name="backlog_balance")
        supply = supply + later - back
    m.add_constraints(supply == xr.DataArray(load, coords=[t]), name="balance")
    m.add_objective(cost)
    status = m.solve(solver_name="highs", output_flag=False)
    return m, status


def price_table(m):
    p = m.constraints["balance"].dual.to_numpy() + 0.0
    return pd.Series(p).round(1).value_counts().sort_index().rename("hours at this price").to_frame().T


normal, _ = week(stress=False)
print(f"normal week with demand response: {normal.objective.value:,.0f} EUR (lesson 7: 9,828,300 EUR); "
      f"demand reduced {normal.variables['reduce'].solution.sum().item():,.0f} MWh")
price_table(normal)

normal week with demand response: 9,828,300 EUR (lesson 7: 9,828,300 EUR); demand reduced 0 MWh


                     0.0   12.0  89.1  94.2
hours at this price    11    18    69    70

In the normal week nothing changes: the price never rises above 94.2 €/MWh, far below the 300 €/MWh at which the first consumers would respond, so no demand is reduced and the cost is exactly lesson 7's. Demand response is an option that is only exercised when power is scarce.

### The stressed week: scarcity

In [4]:
_, status_fixed_demand = week(stress=True, dr=False)
print("stressed week, demand fixed:", status_fixed_demand)
stressed, _ = week(stress=True)
reduced = stressed.variables["reduce"].solution.to_pandas().sum()
shed = stressed.variables["shed"].solution.sum().item()
p_stress = stressed.constraints["balance"].dual.to_numpy() + 0.0
water_value = -stressed.constraints["water"].dual.to_numpy()
print(f"stressed week with demand response: {stressed.objective.value:,.0f} EUR")
print(f"demand reduced: industry {reduced['industry']:,.0f} MWh, commercial {reduced['commercial']:,.0f} MWh; "
      f"involuntary shedding {shed:,.0f} MWh")
print("reservoir water value (EUR/MWh):", np.unique(np.round(water_value, 1)))
price_table(stressed)

stressed week, demand fixed: ('warning', 'infeasible')


stressed week with demand response: 21,045,988 EUR
demand reduced: industry 6,933 MWh, commercial 0 MWh; involuntary shedding 0 MWh
reservoir water value (EUR/MWh): [300.]


                     89.1   128.9  300.0
hours at this price     34     83     51

With demand fixed, the stressed week is **infeasible**: firm capacity is 2,650 MW against a peak of 2,785 MW, and the nearly dry reservoir holds only 6,720 MWh for the whole week. A model without demand response cannot say anything about such a week — it simply fails, as in M2.1 when demand exceeded supply.

With price-responsive demand it solves. Industry cuts 6,933 MWh and the price reaches **300 €/MWh in 51 hours**; commercial consumers and involuntary shedding are not needed. This is **scarcity pricing**: when no generator is left to supply the last MWh, the price is set by the consumer who gives it up — by what that consumer is willing to pay. The reservoir's water value rises to 300 €/MWh as well: every MWh of water now replaces a MWh of industrial demand cut, so the reservoir saves its water for the scarcity hours.

The prices also tell the generators something. In each of the 51 scarcity hours, every MWh the OCGT produces earns 300 − 128.9 = 171.1 €/MWh above its fuel cost. Such scarcity rents are how a plant that runs only in rare tight hours recovers its fixed costs in an energy-only market — and why capping prices well below the value consumers put on electricity can leave too little capacity in the long run.

In [5]:
def pypsa_week(stress=False, shift=None):
    thermal, free, inflow = system(stress)
    n = pypsa.Network()
    n.set_snapshots(h)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=load)
    n.add("Generator", "free", bus="HR", p_nom=free.max(), p_max_pu=free / free.max(), marginal_cost=0)
    for gen, r in thermal.iterrows():
        if r.p_nom > 0:
            n.add("Generator", gen, bus="HR", p_nom=r.p_nom, marginal_cost=r.mc)
    n.add("StorageUnit", "reservoir", bus="HR", p_nom=1000, max_hours=30, p_min_pu=0.0, inflow=inflow,
          cyclic_state_of_charge=True)
    for name, r in DR.iterrows():                       # load reduction offered at the willingness to pay
        n.add("Generator", f"dr_{name}", bus="HR", p_nom=r.p_nom, marginal_cost=r.wtp)
    n.add("Generator", "shedding", bus="HR", p_nom=load.max(), marginal_cost=VOLL)
    if shift:                                           # load shifting = a lossless store of delayed demand
        n.add("StorageUnit", "shifting", bus="HR", p_nom=shift["p"], max_hours=shift["e"] / shift["p"],
              efficiency_store=1.0, efficiency_dispatch=1.0, cyclic_state_of_charge=True)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n


n_stress = pypsa_week(stress=True)
assert np.isclose(n_stress.objective, stressed.objective.value, rtol=1e-9)
assert np.abs(n_stress.buses_t.marginal_price["HR"].to_numpy() - p_stress).max() < 1e-6
print(f"PyPSA, stressed week: {n_stress.objective:,.0f} EUR - same cost and same 168 prices as the own model")

PyPSA, stressed week: 21,045,988 EUR - same cost and same 168 prices as the own model


In PyPSA, price-responsive demand needs no new component: each consumer group is a `Generator` offering load reduction at its willingness to pay, and involuntary shedding is one more at VOLL. (Remember M2.1's sign trap: a demand *bid* entered as a positive `marginal_cost` on a negative generator serves no one. Modelling the *reduction* as a generator avoids it.)

### Load shifting

Heat pumps can pre-heat a building, an electric car can charge later in the night, a cold store can cool down early. Such consumers do not give up consumption — they **move** it. Model a flexible demand of up to 200 MW that can be **delayed** and caught up later, with at most 800 MWh outstanding at any time:

backlog[t] = backlog[t−1] + delay[t] − catch_up[t],  0 ≤ backlog ≤ 800 MWh,  delay, catch_up ≤ 200 MW

Delaying consumption looks to the power system exactly like a storage discharging, and catching up like it charging — with **no losses**. So in PyPSA it is a `StorageUnit` with efficiency 1:

In [6]:
SHIFT = {"p": 200, "e": 800}
results = {}
for stress in (False, True):
    without, _ = week(stress=stress)
    with_shift, _ = week(stress=stress, shift=SHIFT)
    n_shift = pypsa_week(stress=stress, shift=SHIFT)
    assert np.isclose(n_shift.objective, with_shift.objective.value, rtol=1e-9)
    p = with_shift.constraints["balance"].dual.to_numpy() + 0.0
    assert np.abs(n_shift.buses_t.marginal_price["HR"].to_numpy() - p).max() < 1e-6
    results["stressed" if stress else "normal"] = {
        "cost_without": without.objective.value, "cost_with": with_shift.objective.value,
        "saving": without.objective.value - with_shift.objective.value,
        "energy_shifted_MWh": with_shift.variables["delay"].solution.sum().item()}
print("own model = PyPSA (lossless StorageUnit) in both weeks")
pd.DataFrame(results).T.round(0)

own model = PyPSA (lossless StorageUnit) in both weeks


          cost_without   cost_with     saving  energy_shifted_MWh
normal       9828300.0   9543373.0   284927.0              8439.0
stressed    21045988.0  19949837.0  1096151.0              8862.0

In [7]:
print("normal week with shifting:")
print(price_table(week(shift=SHIFT)[0]).to_string())
print("\nstressed week with shifting:")
print(price_table(week(stress=True, shift=SHIFT)[0]).to_string())

normal week with shifting:


                     0.0   12.0  89.1
hours at this price     6    21   141

stressed week with shifting:


                     89.1   128.9  300.0
hours at this price     31     88     49


Load shifting pays in both weeks. In the normal week it saves 284,927 € (2.9 %) by moving 8,439 MWh of consumption: the 70 hours at 94.2 €/MWh disappear entirely — coal is never marginal any more, because evening demand is moved to hours the CCGT can serve — and the hours of curtailed free energy fall from 11 to 6, because delayed demand is caught up when solar is abundant. Compare lesson 7: a 300 MW pumped-storage plant with 2,400 MWh saved 405,156 € in the same week; 200 MW of shifting with an 800 MWh backlog saves about 70 % of that with no losses and no basin to build.

In the stressed week the saving is larger — 1,096,151 € — but the number of scarcity hours falls only from 51 to 49. Shifting moves consumption, it does not remove it: the scarcity lasts for many consecutive evenings, and with at most 800 MWh outstanding the delayed demand must be caught up in hours that are themselves tight. Shifting flattens prices within a day; it cannot replace capacity across a whole stressed week.

## Check your understanding

**Price-responsive demand**

1. Why is a 300 MW load that stops above 300 €/MWh the same, for the model, as a 300 MW generator with a marginal cost of 300 €/MWh? What does the "cost" of that generator measure?
2. In the stressed week the price reaches 300 €/MWh in many hours. Who earns what in those hours — the OCGT, the CCGT, the reservoir? Why might such hours matter for whether those plants stay open?

**Load shifting**

1. Load shifting and lesson 7's pumped hydro both move energy in time. Name two differences that matter for the model and two that matter in reality.
2. The backlog limit (800 MWh) says how much consumption can be outstanding. What would a realistic limit depend on for heat pumps, and for electric cars?